In [1]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

from load_and_check_data import (
    load_residual_load, extract_node_metadata, reconcile_bus_ids,
    check_id_consistency, print_validation_report, assert_ready_for_pipeline,
    build_weighted_adjacency,
)

In [2]:
dataset = pd.read_parquet("dataset/renewables-dataset.parquet")
dataset["Time"] = pd.to_datetime(dataset["Time"])
a_s = 0.05  # solar matches (a_s x 100)% of the average yearly demand across EU
a_w = 0.05  # wind matches (a_w x 100)% of the average yearly demand across EU
dataset_residual = dataset.assign(
    solar_scaled_MWh=lambda df: a_s * df["solar_MWh"],
    wind_scaled_MWh=lambda df: a_w * df["wind_MWh"],
    supply_scaled_MWh=lambda df: df["solar_scaled_MWh"] + df["wind_scaled_MWh"],
    residual_MWh=lambda df: df["demand_MWh"] - df["supply_scaled_MWh"],
    lat = lambda df: df["latitude"],
    long = lambda df: df['longitude'],
    country = lambda df: df["country"]
)[
    [
        "Time",
        "ID",
        "solar_scaled_MWh",
        "wind_scaled_MWh",
        "demand_MWh",
        "supply_scaled_MWh",
        "residual_MWh",
        "latitude",
        "longitude",
        "country"
    ]
]

In [3]:
## checking that parquet matches nodes csv files in terms of bus ids
nodes_df = pd.read_csv("/Users/emrysking/Documents/github-projects/datasci-challenge/dataset/RE-Europe_dataset_package/Static_data/network_nodes.csv")
parquet_meta = extract_node_metadata("/Users/emrysking/Documents/github-projects/datasci-challenge/dataset/renewables-dataset.parquet")
mapping = reconcile_bus_ids(parquet_meta, nodes_df)

Bus ID reconciliation: 1494 parquet buses -> 1494 exact ID matches, 0 matched via coordinates, 0 UNMATCHED


In [4]:
edges_df = pd.read_csv('dataset/RE-Europe_dataset_package/Static_data/network_edges.csv')

In [5]:
## and edges.csv
values, parquet_ids, time_index, report = load_residual_load(
    dataset_residual, time_col="Time", bus_col="ID", value_col="residual_MWh",
)
id_map = dict(zip(mapping.parquet_id, mapping.nodes_csv_id))
canonical_ids = [id_map[pid] for pid in parquet_ids]
check_id_consistency(canonical_ids, "/Users/emrysking/Documents/github-projects/datasci-challenge/dataset/RE-Europe_dataset_package/Static_data/network_edges.csv")

Bus IDs in parquet: 1494, in edges file: 1494
  IDs match exactly between parquet and edges file.


(set(), set())

In [8]:
## adjacency matrices
A, A_norm = build_weighted_adjacency(canonical_ids, edges_df, weight_by="susceptance")

  10 of 2156 edges had 'no data' reactance (X<=1e-05), given a fallback weight of 1.0 instead of Y=1/X
